In [ ]:
##############################################################
# Script to Run Zero Shot, One Shot, and Few Shot Tests with GPT-4o
##############################################################

In [10]:
import os
import json
import random
from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError
from typing import List
from openai import OpenAI
import re
# import torch

# For Hugging Face usage
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

In [11]:
class PaperReview(BaseModel):
    Soundness: int
    Presentation: int
    Contribution: int
    Rating: int
    Confidence: int
    Strengths: str
    Weaknesses: str
    Questions: str

In [12]:
def load_jsonl(file_path: str) -> List[dict]:
    """
    Loads a JSONL file and returns a list of dictionaries.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

def save_jsonl(data: List[dict], file_path: str) -> None:
    """
    Saves a list of dicts to JSONL, one JSON object per line.
    """
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        for d in data:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

def extract_json_string(raw_content: str) -> str:
    """
    Sometimes the model might enclose JSON in triple backticks.
    This regex tries to extract the JSON portion if present.
    Otherwise returns the full raw_content.
    """
    match = re.search(r"```(?:json)?(.*?)```", raw_content, flags=re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        return raw_content.strip()

def parse_paper_review(response_content: str) -> PaperReview:
    """
    Takes the raw string content from a model's response,
    parses it as JSON, and then validates against PaperReview.
    Raises exceptions if invalid.
    """
    try:
        parsed_dict = json.loads(response_content)
        return PaperReview(**parsed_dict)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON returned by the model:\n{e}\nContent:\n{response_content}") from e
    except ValidationError as ve:
        raise ValueError(f"Pydantic validation error:\n{ve}\nContent:\n{response_content}") from ve



In [13]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def run_gpt4o_inference(messages: List[dict]) -> str:
    """
    Given a list of messages in the OpenAI chat format, 
    calls the GPT-4 'o' model (whatever alias you have) 
    and returns the text content of the top choice.
    """
    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )
        raw_content = completion.choices[0].message.content
        json_str = extract_json_string(raw_content)
        review = parse_paper_review(json_str)
        print("Zero shot review:\n", review)
        return review
    except Exception as e:
        print("Zero-shot error:", e)

    

In [16]:
zero_shot_files = [
    #"data/test_data/zero_shot/test_data_2024_abstract_prompts.jsonl",
    #"data/test_data/zero_shot/test_data_2024_full text_prompts.jsonl",
    #"data/test_data/zero_shot/test_data_2024_summary_prompts.jsonl",
    #"data/test_data/zero_shot/test_data_2025_abstract_prompts.jsonl",
    #"data/test_data/zero_shot/test_data_2025_full text_prompts.jsonl",
    #"data/test_data/zero_shot/test_data_2025_summary_prompts.jsonl",
]
one_shot_files = [
    #"data/test_data/one_shot/test_data_2024_abstract_prompts_oneshot.jsonl",
    #"data/test_data/one_shot/test_data_2024_full text_prompts_oneshot.jsonl",
    #"data/test_data/one_shot/test_data_2024_summary_prompts_oneshot.jsonl",
    #"data/test_data/one_shot/test_data_2025_abstract_prompts_oneshot.jsonl",
    #"data/test_data/one_shot/test_data_2025_full text_prompts_oneshot.jsonl",
    #"data/test_data/one_shot/test_data_2025_summary_prompts_oneshot.jsonl",
]
few_shot_files = [
    #"data/test_data/few_shot/test_data_2024_abstract_prompts_fewshot.jsonl",
    "data/test_data/few_shot/test_data_2024_full text_prompts_fewshot.jsonl",
    #"data/test_data/few_shot/test_data_2024_summary_prompts_fewshot.jsonl",
    #"data/test_data/few_shot/test_data_2025_abstract_prompts_fewshot.jsonl",
    "data/test_data/few_shot/test_data_2025_full text_prompts_fewshot.jsonl",
    #"data/test_data/few_shot/test_data_2025_summary_prompts_fewshot.jsonl",
]

all_prompt_files = zero_shot_files + one_shot_files + few_shot_files

print("=== Running GPT-4o Inference ===")
evaluate_model_on_files(
    model_name="gpt4o",
    model_inference_func=run_gpt4o_inference,
    input_files=all_prompt_files,
    output_dir="results/gpt4o"
)

=== Running GPT-4o Inference ===
MESSAGE IS:
[{'role': 'system', 'content': 'You are a helpful assistant that extracts a structured summary from a paper. Please return only valid JSON with the following fields:\nSoundness, Presentation, Contribution, Rating, Confidence (all integers)\nStrengths, Weaknesses, and Questions (all strings).\nNo additional keys. No extra text.'}, {'role': 'user', 'content': 'Below are example prompts & responses:\n\nEXAMPLE 1 (USER):\nPlease read the following paper and produce reviewer scores.\n\n--- PAPER ---\n\n\nCROSS-MODALITY DEBIASING: USING LANGUAGE TO\nMITIGATE SUB-POPULATION SHIFTS IN IMAGING\n\nAnonymous authors\nPaper under double-blind review\n\nABSTRACT\n\nSub-population shift is a specific type of domain shift that highlights changes in\ndata distribution within specific sub-groups or populations between training and\ntesting. Sub-population shift accounts for a significant source of algorithmic bias\nand calls for distributional robustness. Re

KeyboardInterrupt: 